In [1]:
%matplotlib inline

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from matplotlib import rcParams
from ipywidgets import IntSlider, interact

In [3]:
rcParams['font.family'] = 'SimHei'
rcParams['axes.unicode_minus'] = False

In [83]:
df = pd.read_csv("../data/raw/may_filtered.csv")
datetimestr = df['date'] + ' ' + df['time']
df['datetime'] = pd.to_datetime(datetimestr, format='%d-%m-%Y %H:%M:%S').astype("datetime64[s]")
df['duration(s)'] = pd.to_timedelta(df.duration).dt.total_seconds()
df.drop(['date', 'time', 'duration'], inplace=True, axis=1)
df['motion'] = np.linalg.norm(df[['accX', 'accY', 'accZ']].values, axis=1)
df = df[["datetime", "duration(s)", "red", "ied", "motion"]]

In [82]:
pd.to_timedelta(df.duration[:500]).dt.total_seconds()


0      0.0
1      0.0
2      0.0
3      0.0
4      0.0
      ... 
495    2.0
496    2.0
497    2.0
498    2.0
499    2.0
Name: duration, Length: 500, dtype: float64

In [54]:
a = pd.to_timedelta(pd.Series(df.duration.unique())).diff().dt.total_seconds().fillna(0).astype(int)

In [65]:
c = pd.to_timedelta(pd.Series(df.duration.unique())).dt.total_seconds()

In [69]:
np.where(c==0)[0]

array([0])

In [71]:
import pandas as pd
import numpy as np

# --- 前置准备：将字符串类型的 duration 转为时间格式 ---
# 加上 errors='coerce'，遇到无法解析的异常字符串（比如纯文字）会转成 NaT
df['duration_dt'] = pd.to_datetime(df['duration'], errors='coerce')

# --- 任务一：找出 duration 突然变成 0 或重置的 Index ---

# 情况 A：字面意义上的等于 '0' 或者是 '00:00:00'
zero_mask = df['duration'].astype(str).str.strip().isin(['0', '0.0', '00:00:00'])
zero_indices = df[zero_mask].index.tolist()

# 情况 B：发生“负跳变”（时间倒流，往往是计时器重启的标志）
# 计算相邻两行的秒数差
duration_diff_seconds = df['duration_dt'].diff().dt.total_seconds()
# 如果差值小于 0，说明计时器往回跳了
reset_mask = duration_diff_seconds < 0
reset_indices = df[reset_mask].index.tolist()

print(f"🔍 发现 duration 字面量为 0 的记录数: {len(zero_indices)} 条")
if len(zero_indices) > 0:
    print(f"   前几个 index 为: {zero_indices[:5]}")

print(f"🔄 发现 duration 发生负跳变（重启回拨）的记录数: {len(reset_indices)} 条")
if len(reset_indices) > 0:
    print(f"   前几个 index 为: {reset_indices[:5]}\n")


# --- 任务二：对比 datetime 和 duration 的断联间隔是否一致 ---

# 计算 datetime 的相邻秒数差
datetime_diff_seconds = df['datetime'].diff().dt.total_seconds()

# 筛选出真正发生“断点”的行 (正常跨秒是 1 秒，所以我们找 > 1.5 秒的)
gap_mask = datetime_diff_seconds > 1.5

# 提取这些发生断点的行，对比两个时间列的差值
compare_df = pd.DataFrame({
    'gap_datetime (秒)': datetime_diff_seconds[gap_mask],
    'gap_duration (秒)': duration_diff_seconds[gap_mask]
})

# 计算两者的误差 (如果绝对一致，误差应该为 0)
compare_df['误差'] = compare_df['gap_datetime (秒)'] - compare_df['gap_duration (秒)']

print("-" * 60)
print(f"📊 总共检测到 {len(compare_df)} 次大于 1.5 秒的断点。")
print("下面是 datetime 和 duration 断联时间的对比结果前 10 条：")
print(compare_df.head(10))

# 检查有没有误差非常大的断联 (容忍 100Hz 带来的 0.01 秒左右的微小计算误差)
mismatch_df = compare_df[compare_df['误差'].abs() > 0.1]
if len(mismatch_df) > 0:
    print(f"\n⚠️ 警告：发现 {len(mismatch_df)} 处断点发生时，datetime 和 duration 断开的时长不一致！")
else:
    print("\n✅ 好消息：在所有断联期间，datetime 和 duration 的中断时长完全吻合。")

C:\Users\wangsx\AppData\Local\Temp\ipykernel_30908\2688466528.py:6: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['duration_dt'] = pd.to_datetime(df['duration'], errors='coerce')


🔍 发现 duration 字面量为 0 的记录数: 0 条
🔄 发现 duration 发生负跳变（重启回拨）的记录数: 4 条
   前几个 index 为: [1156576, 2174742, 2597883, 5886579]

------------------------------------------------------------
📊 总共检测到 230 次大于 1.5 秒的断点。
下面是 datetime 和 duration 断联时间的对比结果前 10 条：
         gap_datetime (秒)  gap_duration (秒)   误差
36780               591.0             591.0  0.0
142160               24.0              23.0  1.0
829558                2.0               2.0  0.0
829858                2.0               2.0  0.0
830138               26.0              26.0  0.0
895556               63.0              63.0  0.0
1100992             155.0             155.0  0.0
1101414               2.0               2.0  0.0
1103554              64.0              64.0  0.0
1111516              45.0              45.0  0.0

⚠️ 警告：发现 121 处断点发生时，datetime 和 duration 断开的时长不一致！


In [73]:
display(df[['datetime', 'duration']].iloc[1156576 - 5 : 1156576 + 5])

,datetime,duration
1156571,2026-05-02 18:18:04,1:56:19
1156572,2026-05-02 18:18:04,1:56:19
1156573,2026-05-02 18:18:04,1:56:19
1156574,2026-05-02 18:18:04,1:56:19
1156575,2026-05-02 18:18:04,1:56:19
1156576,2026-05-02 18:30:51,0:00:00
1156577,2026-05-02 18:30:51,0:00:00
1156578,2026-05-02 18:30:51,0:00:00
1156579,2026-05-02 18:30:51,0:00:00
1156580,2026-05-02 18:30:51,0:00:00
